## Scoring Random Forest and CNN-LSTM Baselines with ClimateBench NRMSE

- Computed over 2080-2100
- **Spatial NRMSE**: error of the 2080–2100 mean map, divided by the true global-mean change
- **Global NRMSE**: error of the yearly global mean, divided by the same number
- **Total** = Spatial + 5 × Global

In [19]:
import os, glob
import numpy as np
import pandas as pd
import xarray as xr
import json

In [4]:
data_path = os.path.expanduser('~/climatebench/data/')
rf_dir    = os.path.expanduser('~/climatebench/xiang_week3/rf_results/')
cnn_dir   = os.path.expanduser('~/climatebench/xiang_week3/cnn_results/')
out_dir   = os.path.expanduser('~/climatebench/xiang_week3/')
os.makedirs(out_dir, exist_ok=True)

SECONDS_PER_DAY = 86400

#seeds from cnn
SEEDS = [0, 1, 2, 3, 4] 

#scoring window
START, END = 2080, 2100

VARS = {"tas": "tas", "dtr": "diurnal_temperature_range", "pr": "pr", "pr90": "pr90"}
UNITS = {"tas": "K", "dtr": "K", "pr": "mm/day", "pr90": "mm/day"}

**load ssp245 output (truth) averaged over ensemble members**

In [5]:
truth = xr.open_dataset(data_path + 'outputs_ssp245.nc').mean('member').sel(time=slice(2015, 2100)).compute()
truth['pr']   = truth['pr'] * SECONDS_PER_DAY
truth['pr90'] = truth['pr90'] * SECONDS_PER_DAY
print(dict(truth.sizes))

{'time': 86, 'lat': 96, 'lon': 144}


**functions to calculate metrics**

In [6]:
def global_mean(da):
    w = np.cos(np.deg2rad(da.lat))
    return da.weighted(w).mean(['lat', 'lon'])

def scores(true, pred, start=START, end=END):
    t = true.sel(time=slice(start, end))
    p = pred.sel(time=slice(start, end))
    norm = abs(float(global_mean(t.mean('time'))))          # true global-mean change
    rmse_map = float(np.sqrt(global_mean((t.mean('time') - p.mean('time')) ** 2)))
    rmse_glb = float(np.sqrt(((global_mean(t) - global_mean(p)) ** 2).mean('time')))
    return {"RMSE (mean map)": rmse_map,
            "NRMSE spatial": rmse_map / norm,
            "NRMSE global": rmse_glb / norm,
            "NRMSE total": rmse_map / norm + 5 * rmse_glb / norm}

def wrap(pred_np, var):
    """(86, 96, 144) numpy -> DataArray with the truth's time/lat/lon."""
    assert pred_np.shape == truth[var].shape, (pred_np.shape, truth[var].shape)
    return xr.DataArray(pred_np, coords=truth[var].coords, dims=truth[var].dims)

**load model predictions**

In [13]:
preds = {}

#rf preds: one per var
for short, var in VARS.items():
    f = os.path.join(rf_dir, f"pred_{short}.npy")
    if os.path.exists(f):
        preds[("Random Forest", short)] = wrap(np.load(f), var)
        print(f'rf: {short}')
    else:
        print("missing RF file:", f)

#cnn-lstm preds: one per seed + avg over seeds
cnn_seed_preds = {}
for short, var in VARS.items():
    arrs = {}
    for s in SEEDS:
        f = os.path.join(cnn_dir, f"pred_{var}_seed{s}.npy")
        if os.path.exists(f):
            arrs[s] = np.load(f)
    if not arrs:
        print("no CNN predictions for", var)
        continue
    cnn_seed_preds[short] = {s: wrap(a, var) for s, a in arrs.items()}
    preds[("CNN-LSTM (seed mean)", short)] = wrap(np.mean(list(arrs.values()), axis=0), var)
    print(f"CNN {short}: seeds {sorted(arrs)}")

rf: tas
rf: dtr
rf: pr
rf: pr90
CNN tas: seeds [0, 1, 2, 3, 4]
CNN dtr: seeds [0, 1, 2, 3, 4]
CNN pr: seeds [0, 1, 2, 3, 4]
CNN pr90: seeds [0, 1, 2, 3, 4]


**calculate model scores**

In [14]:
rows = []
for (model, short), pred in preds.items():
    rows.append({"model": model, "variable": short, **scores(truth[VARS[short]], pred)})

score_df = (pd.DataFrame(rows)
              .set_index(["model", "variable"])
              .reindex(VARS.keys(), level="variable"))
score_df.to_csv(os.path.join(out_dir, "scores_2080_2100.csv"))
score_df.round(3)

RMSE (mean map)  NRMSE spatial  NRMSE global  \
model                variable                                                 
Random Forest        tas                 0.179          0.086         0.163   
                     dtr                 0.160         14.279         2.947   
                     pr                  0.180          2.764         0.511   
                     pr90                0.483          3.125         0.583   
CNN-LSTM (seed mean) tas                 0.201          0.097         0.071   
                     dtr                 0.088          7.890         1.359   
                     pr                  0.129          1.972         0.240   
                     pr90                0.397          2.568         0.312   

                               NRMSE total  
model                variable               
Random Forest        tas             0.900  
                     dtr            29.014  
                     pr              5.317  
                     pr90            6.039  
CNN-LSTM (seed mean) tas             0.453  
                     dtr            14.683  
                     pr              3.170  
                     pr90            4.127

**compare with climatebench scores**

In [17]:
paper = pd.DataFrame(
    [["Random Forest",  "tas", 0.108, 0.058, 0.400], ["Random Forest",  "dtr", 9.195, 2.652, 22.457],
     ["Random Forest",  "pr",  2.524, 0.502, 5.035], ["Random Forest",  "pr90", 2.682, 0.543, 5.399],
     ["Neural Network", "tas", 0.107, 0.044, 0.327], ["Neural Network", "dtr", 9.917, 1.372, 16.778],
     ["Neural Network", "pr",  2.128, 0.209, 3.175], ["Neural Network", "pr90", 2.610, 0.346, 4.339]],
    columns=["model", "variable", "NRMSE spatial", "NRMSE global", "NRMSE total"])

name_map = {"Random Forest": "Random Forest", "CNN-LSTM (seed mean)": "Neural Network"}
ours = (score_df.reset_index()
                .assign(model=lambda d: d.model.map(name_map))
                [["model", "variable", "NRMSE spatial", "NRMSE global", "NRMSE total"]])

comparison = ours.merge(paper, on=["model", "variable"], suffixes=(" (ours)", " (climatebench)"))
comparison = comparison[["model", "variable",
                         "NRMSE spatial (ours)", "NRMSE spatial (climatebench)",
                         "NRMSE global (ours)", "NRMSE global (climatebench)",
                         "NRMSE total (ours)", "NRMSE total (climatebench)"]]
comparison.to_csv(os.path.join(out_dir, "comparison_with_paper.csv"), index=False)
comparison.round(3)

,model,variable,NRMSE spatial (ours),NRMSE spatial (climatebench),NRMSE global (ours),NRMSE global (climatebench),NRMSE total (ours),NRMSE total (climatebench)
0,Random Forest,tas,0.086,0.108,0.163,0.058,0.900,0.400
1,Random Forest,dtr,14.279,9.195,2.947,2.652,29.014,22.457
2,Random Forest,pr,2.764,2.524,0.511,0.502,5.317,5.035
3,Random Forest,pr90,3.125,2.682,0.583,0.543,6.039,5.399
4,Neural Network,tas,0.097,0.107,0.071,0.044,0.453,0.327
5,Neural Network,dtr,7.890,9.917,1.359,1.372,14.683,16.778
6,Neural Network,pr,1.972,2.128,0.240,0.209,3.170,3.175
7,Neural Network,pr90,2.568,2.610,0.312,0.346,4.127,4.339


**Task 3: Random forest best hyperparameters**

In [20]:
best_params = {}
for short in ["tas", "dtr", "pr", "pr90"]:
    f = os.path.join(rf_dir, f"best_params_{short}.json")
    if os.path.exists(f):
        with open(f) as fh:
            best_params[short] = json.load(fh)
    else:
        print("missing:", f)

params_df = pd.DataFrame(best_params).T 
params_df.index.name = "variable"
params_df.to_csv(os.path.join(out_dir, "rf_best_params.csv"))
params_df

,n_estimators,min_samples_split,min_samples_leaf,max_features,max_depth,bootstrap
variable,,,,,,
tas,250,10,4,1.0,50,True
dtr,150,5,8,sqrt,None,True
pr,250,25,12,1.0,45,True
pr90,250,10,8,1.0,45,True


**Task 4: Spread across random seeds**

In [18]:
seed_rows = []
for short, by_seed in cnn_seed_preds.items():
    for s, pred in by_seed.items():
        seed_rows.append({"variable": short, "seed": s, **scores(truth[VARS[short]], pred)})

seed_df = pd.DataFrame(seed_rows)
seed_df.to_csv(os.path.join(out_dir, "cnn_scores_per_seed.csv"), index=False)

metric_cols = ["NRMSE spatial", "NRMSE global", "NRMSE total", "RMSE (mean map)"]
spread = (seed_df.groupby("variable")[metric_cols]
                 .agg(["mean", "std", "min", "max"])
                 .reindex(VARS.keys()))
spread.to_csv(os.path.join(out_dir, "cnn_seed_spread.csv"))

for short in spread.index:
    m, s = spread.loc[short, ("NRMSE total", "mean")], spread.loc[short, ("NRMSE total", "std")]
    n = int((seed_df.variable == short).sum())
    print(f"{short:5s} total NRMSE: {m:.3f} ± {s:.3f} across {n} seeds")

spread.round(3)

tas   total NRMSE: 0.600 ± 0.223 across 5 seeds
dtr   total NRMSE: 15.389 ± 1.657 across 5 seeds
pr    total NRMSE: 3.279 ± 0.115 across 5 seeds
pr90  total NRMSE: 4.481 ± 0.155 across 5 seeds


NRMSE spatial                      NRMSE global                       \
                  mean    std    min    max         mean    std    min    max   
variable                                                                        
tas              0.134  0.039  0.108  0.202        0.093  0.037  0.071  0.158   
dtr              8.352  0.299  7.894  8.668        1.407  0.347  0.895  1.825   
pr               2.079  0.078  1.949  2.148        0.240  0.013  0.226  0.258   
pr90             2.853  0.245  2.523  3.180        0.326  0.022  0.305  0.349   

         NRMSE total                        RMSE (mean map)                \
                mean    std     min     max            mean    std    min   
variable                                                                    
tas            0.600  0.223   0.461   0.991           0.278  0.081  0.224   
dtr           15.389  1.657  13.143  17.362           0.094  0.003  0.088   
pr             3.279  0.115   3.124   3.417           0.136  0.005  0.127   
pr90           4.481  0.155   4.268   4.706           0.441  0.038  0.390   

                 
            max  
variable         
tas       0.419  
dtr       0.097  
pr        0.140  
pr90      0.492